<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 30 · Portfolio Valuation

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
import sys
import math
import datetime as dt
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

# Add the code directory to sys.path so dxlib can be imported
CODE_DIR = (Path("..") / "code").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from typing import Any, Protocol
from dxlib import *

class Pricer(Protocol):
    def value(self, spot: float) -> tuple[float, float]: ...
    
snapshot_path = Path("..") / "data" / "spx_options_snapshot.csv"
snap = load_spx_snapshot(snapshot_path) if snapshot_path.exists() else None
if snap is not None:
    short_expiry = dt.date(2026, 3, 20)
    long_expiries = [dt.date(2026, 6, 18), dt.date(2026, 12, 18)]
    short_surface_df = select_small_surface(snap, [short_expiry], rate=0.03)
    long_surface_df = select_small_surface(snap, long_expiries, rate=0.03)
    surface_df = long_surface_df
    paths = 15_000
    steps_per_year = 320
    rate = 0.03
    h_params = HestonParams(
        kappa=2.6,
        theta=0.046,
        vol_of_vol=0.9,
        rho=-0.67,
        v0=0.018,
    )
    params = h_params
    local_map = {
        dt.date(2026, 6, 18): (0.0606, 0.0128, -0.732),
        dt.date(2026, 12, 18): (0.0484, 0.0149, -0.763),
    }
    seed = 11

## Why Portfolios Need Structure

Execute the code examples below.


## A Minimal Pricing Contract

Execute the code examples below.


In [ ]:
def price_and_stderr(pricer, spot):
    raw = pricer.value(spot)
    if isinstance(raw, tuple) and len(raw) == 2:
        price, stderr = raw
        return float(price), float(stderr)
    if isinstance(raw, dict):
        price = raw.get("price")
        stderr = raw.get("stderr")
        return float(price), float(stderr)
    raise TypeError("Unsupported pricer return type")

## Positions and Aggregation

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class Position:
    name: str
    quantity: float
    pricer: Pricer

    def value(self, spot):
        price, stderr = price_and_stderr(self.pricer, spot)
        value = float(self.quantity) * price
        err = abs(float(self.quantity)) * stderr
        return value, err

## Bump-and-Revalue Delta

Execute the code examples below.


In [ ]:
def delta_central(pricer, spot, *, rel_bump=0.01):
    bump = float(spot) * float(rel_bump)
    up, _ = price_and_stderr(pricer, spot + bump)
    down, _ = price_and_stderr(pricer, spot - bump)
    return float((up - down) / (2.0 * bump))

## Diagnostic Figures (Portfolio Contributions)

Execute the code examples below.


## Interactive Session: Building and Valuing a Small Book

Execute the code examples below.


In [ ]:
import pandas as pd

from dxlib import (
    AmericanPut,
    AmericanPutLSM,
    EuropeanCall,
    EuropeanMCPricer,
    EuropeanPut,
    FlatDiscounting,
    GeometricBrownianMotion,
    Portfolio,
    Position,
)

In [ ]:
s0 = 36.0

r = 0.06

sigma = 0.2

ttm = 1.0

steps = 50

disc = FlatDiscounting(rate=r)

gbm_q = GeometricBrownianMotion(
    drift=r,
    volatility=sigma,
    seed=7,
)

In [ ]:
euro_put = EuropeanPut(strike=40.0)

euro_call = EuropeanCall(strike=40.0)

euro_put_pricer = EuropeanMCPricer(
    process=gbm_q,
    payoff=euro_put,
    discounting=disc,
    maturity=ttm,
    steps=steps,
    paths=200_000,
)

euro_call_pricer = EuropeanMCPricer(
    process=gbm_q,
    payoff=euro_call,
    discounting=disc,
    maturity=ttm,
    steps=steps,
    paths=200_000,
)

am_put_pricer = AmericanPutLSM(
    process=gbm_q,
    payoff=AmericanPut(strike=40.0),
    discounting=disc,
    maturity=ttm,
    steps=steps,
    paths=200_000,
    basis_degree=2,
)

In [ ]:
positions = (
    Position(
        name="American put (K=40)",
        quantity=1.0,
        pricer=am_put_pricer,
    ),
    Position(
        name="Short European put (K=40)",
        quantity=-2.0,
        pricer=euro_put_pricer,
    ),
    Position(
        name="European call (K=40)",
        quantity=1.0,
        pricer=euro_call_pricer,
    ),
)

book = Portfolio(
    name="Index options book",
    positions=positions,
)

In [ ]:
report = book.value(spot=s0)

df = pd.DataFrame(report["positions"]).set_index("name")

df[["quantity", "value", "stderr"]].round(4)

In [ ]:
deltas = book.deltas(spot=s0, rel_bump=0.01)

for k in deltas:
    print(k, round(deltas[k], 4))

## Where We Are Heading Next

Execute the code examples below.


## Appendix: `dxlib` Portfolio Source Code

Execute the code examples below.


## `code/dxlib/portfolio.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 30 - Portfolio Valuation.

Portfolio containers and simple bump-and-revalue sensitivities.

(c) Dr. Yves J. Hilpisch
AI-supported by GPT 5.x
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any, Protocol

__all__ = [
    "Pricer",
    "Position",
    "Portfolio",
    "price_and_stderr",
    "delta_central",
]


class Pricer(Protocol):
    """
    Minimal pricing interface used by `dxlib.portfolio`.

    The `value()` method may return either:

    - a `(price, stderr)` tuple, or
    - a dictionary with keys `"price"` and `"stderr"`.
    """

    def value(self, spot: float) -> Any: ...


def price_and_stderr(pricer: Pricer, spot: float) -> tuple[float, float]:
    """
    Normalize different pricer return types to `(price, stderr)`.
    """

    raw = pricer.value(spot)
    if isinstance(raw, tuple) and len(raw) == 2:
        price, stderr = raw
        return float(price), float(stderr)
    if isinstance(raw, dict):
        price = raw.get("price")
        stderr = raw.get("stderr")
        if price is None or stderr is None:
            raise ValueError(
                "dict result must contain keys 'price' and 'stderr'"
            )
        return float(price), float(stderr)
    raise TypeError("Unsupported pricer return type")


def delta_central(
    pricer: Pricer,
    spot: float,
    *,
    rel_bump: float = 0.01,
) -> float:
    """
    Central-difference delta estimate via bump-and-revalue.
    """

    if spot <= 0:
        raise ValueError("spot must be positive")
    if rel_bump <= 0:
        raise ValueError("rel_bump must be positive")

    bump = float(spot) * float(rel_bump)
    up, _ = price_and_stderr(pricer, spot + bump)
    down, _ = price_and_stderr(pricer, spot - bump)
    return float((up - down) / (2.0 * bump))


@dataclass(frozen=True, slots=True)
class Position:
    """
    A portfolio position defined by a quantity and a pricer.
    """

    name: str
    quantity: float
    pricer: Pricer

    def value(self, spot: float) -> tuple[float, float]:
        price, stderr = price_and_stderr(self.pricer, spot)
        return float(self.quantity) * price, abs(float(self.quantity)) * stderr

    def delta(self, spot: float, *, rel_bump: float = 0.01) -> float:
        return float(self.quantity) * delta_central(
            self.pricer,
            spot,
            rel_bump=rel_bump,
        )


@dataclass(frozen=True, slots=True)
class Portfolio:
    """
    A collection of positions with simple aggregation logic.
    """

    name: str
    positions: tuple[Position, ...]

    def value(self, spot: float) -> dict[str, object]:
        values: list[dict[str, float]] = []
        total_value = 0.0
        total_var = 0.0
        for pos in self.positions:
            pos_value, pos_stderr = pos.value(spot)
            total_value += pos_value
            total_var += pos_stderr**2
            values.append(
                {
                    "name": pos.name,
                    "value": float(pos_value),
                    "stderr": float(pos_stderr),
                    "quantity": float(pos.quantity),
                }
            )

        total_stderr = float(math.sqrt(total_var))
        return {
            "name": self.name,
            "spot": float(spot),
            "positions": values,
            "total_value": float(total_value),
            "total_stderr": total_stderr,
            "stderr_method": "independent_root_sum_squares",
        }

    def deltas(
        self,
        spot: float,
        *,
        rel_bump: float = 0.01,
    ) -> dict[str, float]:
        out: dict[str, float] = {}
        for pos in self.positions:
            out[pos.name] = pos.delta(spot, rel_bump=rel_bump)
        out["portfolio"] = float(sum(out.values()))
        return out

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
